<a href="https://colab.research.google.com/github/justAnormlaguy/Inteligencia-Artificial/blob/main/Trabalho_Aula001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


SHEET_ID = "18QzIgT1gOx_jWlwjyIAigD-HG2z4qjJNFRFb7evQUhc"
GID = "0"  # aba Página1

url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"

# Lendo com tabulação e vírgula decimal
df = pd.read_csv(url_tsv, sep="\t", decimal=",")
print("Colunas lidas:", list(df.columns))

# 2. Configuração de Variáveis (Features e Target)
FEATURES = ["peso_g", "diametro_cm"]
TARGET = "fruta"

# Convertendo para numérico e tratando o target
for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df = df.dropna(subset=FEATURES + [TARGET]).copy()

# Encode do Target
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df[TARGET])
print("\nClasses identificadas:", list(le.classes_))

# 3. Divisão Treino e Teste (80% / 20% manual como no original)
RANDOM_STATE = 42
df_shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

limite_teste = int(len(df) * 0.20)
test_df  = df_shuffled.iloc[:limite_teste].copy()
train_df = df_shuffled.iloc[limite_teste:len(df)].copy()

X_train = train_df[FEATURES].values
y_train = train_df['target_encoded'].values
X_test  = test_df[FEATURES].values
y_test  = test_df['target_encoded'].values

print(f"\nTotal: {len(df)} | Treino: {len(train_df)} | Teste: {len(test_df)}")

# 4. Criação dos Pipelines
# Pipeline KNN
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=3))
])

# Pipeline Árvore de Decisão (Decision Tree não exige scaler, mas mantemos no pipeline por padronização)
pipeline_tree = Pipeline([
    ('scaler', StandardScaler()),
    ('tree', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

# 5. Treinamento e Avaliação
print("\n" + "="*40)
print(" RESULTADOS: K-NEAREST NEIGHBORS (KNN)")
print("="*40)
pipeline_knn.fit(X_train, y_train)
y_pred_knn = pipeline_knn.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, y_pred_knn):.2f}")
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_knn))
print("Relatório de Classificação:\n", classification_report(y_test, y_pred_knn, target_names=le.classes_))


print("\n" + "="*40)
print(" RESULTADOS: ÁRVORE DE DECISÃO")
print("="*40)
pipeline_tree.fit(X_train, y_train)
y_pred_tree = pipeline_tree.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, y_pred_tree):.2f}")
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_tree))
print("Relatório de Classificação:\n", classification_report(y_test, y_pred_tree, target_names=le.classes_))

Colunas lidas: ['peso_g', 'diametro_cm', 'fruta']

Classes identificadas: ['laranja', 'maca']

Total: 15 | Treino: 12 | Teste: 3

 RESULTADOS: K-NEAREST NEIGHBORS (KNN)
Acurácia: 1.00
Matriz de Confusão:
 [[2 0]
 [0 1]]
Relatório de Classificação:
               precision    recall  f1-score   support

     laranja       1.00      1.00      1.00         2
        maca       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3


 RESULTADOS: ÁRVORE DE DECISÃO
Acurácia: 1.00
Matriz de Confusão:
 [[2 0]
 [0 1]]
Relatório de Classificação:
               precision    recall  f1-score   support

     laranja       1.00      1.00      1.00         2
        maca       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1